# Cross-Sectional Regression & Coverage Volume

Tests whether newspaper owners with railroad financial ties produced more anti-labor coverage.
Runs the main cross-sectional specification and coverage volume regressions.

In [12]:
import pandas as pd
import sqlite3
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

DB_PATH = '../data/processed/newspapers.db'
OE_PATH = '../data/intermediate/personnel_coding/owners_and_editors.csv'
CODE_PATH = '../data/intermediate/personnel_coding/combined_coding.csv'

# Load intermediate data from 01_data_preparation
df = pd.read_csv('intermediate/analysis_sample.csv')
counts = pd.read_csv('intermediate/sentiment_counts.csv')
paper_year_rr = pd.read_csv('intermediate/paper_year_rr.csv')
person_rr = pd.read_csv('intermediate/person_rr.csv')


oe = pd.read_csv(OE_PATH)

print(f'Loaded analysis sample: {len(df)} newspaper-years')

Loaded analysis sample: 319 newspaper-years


## Regression

In [13]:
# Prepare data
df['year_str'] = df['year'].astype(str)

# Build circulation lookup from master.csv (nearest Ayer directory year)
import numpy as np
master = pd.read_csv('../data/processed/master.csv', low_memory=False)
circ_cols = [c for c in master.columns if 'circulation' in c.lower()]
circ_long = []
for col in circ_cols:
    yr = int(col.split()[0])
    tmp = master[['issn', col]].dropna(subset=['issn', col]).copy()
    tmp = tmp.rename(columns={col: 'circulation'})
    tmp['circ_year'] = yr
    circ_long.append(tmp)
circ_long = pd.concat(circ_long, ignore_index=True)
circ_long['circulation'] = pd.to_numeric(circ_long['circulation'], errors='coerce')
circ_long = circ_long.dropna(subset=['circulation'])
circ_long = circ_long[circ_long['circulation'] > 0]

circ_lookup = {}
for _, row in circ_long.iterrows():
    circ_lookup.setdefault(row['issn'], []).append((row['circ_year'], row['circulation']))

def nearest_circ(issn, year):
    entries = circ_lookup.get(issn)
    if not entries:
        return None
    return min(entries, key=lambda x: abs(x[0] - year))[1]

df['circulation'] = df.apply(lambda r: nearest_circ(r['issn'], r['year']), axis=1)
df['log_circulation'] = np.log(pd.to_numeric(df['circulation'], errors='coerce'))

print(f"Circulation available for {df['log_circulation'].notna().sum()} / {len(df)} obs")

# Sanity check: 10 newspapers with county population
town_lookup = master.set_index('issn')['town'].to_dict()
check = (df.drop_duplicates('issn')[['issn', 'state', 'county_pop', 'log_county_pop']]
         .assign(town=lambda x: x['issn'].map(town_lookup))
         .dropna(subset=['log_county_pop'])
         .head(10)[['town', 'state', 'county_pop', 'log_county_pop']])
print("\nCounty population sanity check:")
print(check.to_string(index=False))

Circulation available for 191 / 319 obs

County population sanity check:
          town                state  county_pop  log_county_pop
       Belfast                MAINE     34316.1       10.443370
      New York             NEW YORK    942292.0       13.756070
    Saint Paul                  NaN     41329.0       10.629320
    FORT WORTH                TEXAS     41142.0       10.624785
    Alexandria             VIRGINIA     16755.0        9.726452
    Sacramento           CALIFORNIA     40339.0       10.605074
    WASHINGTON DISTRICT OF COLUMBIA    198731.2       12.199708
      Richmond             VIRGINIA     66179.0       11.100118
Salt Lake City                  NaN     45217.0       10.719228
     Lancaster         PENNSYLVANIA    139447.0       11.845440


### Cross-Sectional Regression (newspaper-level)

Collapses all years into a single observation per newspaper: total anti-labor / total labor articles across the owner's full tenure. N = number of newspapers, not newspaper-years.

In [14]:
# Collapse to one observation per newspaper (sum articles across all years)
df_xs = (
    df.groupby('issn')
    .agg(anti_labor=('anti_labor', 'sum'),
         total_labor=('total_labor', 'sum'),
         railroad_interest=('railroad_interest', 'max'),
         person_republican=('person_republican', 'max'),
         log_circulation=('log_circulation', 'mean'),
         log_county_pop=('log_county_pop', 'mean'))
    .reset_index()
)
df_xs['anti_labor_intensity'] = df_xs['anti_labor'] / df_xs['total_labor']

# OLS (HC3)
m0a = smf.ols('anti_labor_intensity ~ railroad_interest', data=df_xs).fit(cov_type='HC3')

df_xs_pol = df_xs.dropna(subset=['person_republican'])
m0b = smf.ols('anti_labor_intensity ~ railroad_interest + person_republican',
              data=df_xs_pol).fit(cov_type='HC3')

df_xs_pop = df_xs.dropna(subset=['person_republican', 'log_county_pop'])
m0d = smf.ols('anti_labor_intensity ~ railroad_interest + person_republican + log_county_pop',
              data=df_xs_pop).fit(cov_type='HC3')

from statsmodels.iolib.summary2 import summary_col
print("Cross-sectional regressions — OLS (HC3 SEs)")
print()
print(summary_col(
    [m0a, m0b, m0d],
    model_names=['(0a) Bivariate', '(0b) + Party', '(0c) + Party + Pop'],
    stars=True,
    info_dict={'N': lambda m: m.nobs, 'R²': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'person_republican', 'log_county_pop', 'Intercept']
))

# WLS (weighted by total_labor, HC3)
print("\n\nCross-sectional regressions — WLS weighted by total_labor (HC3 SEs)")
print()
wxa = smf.wls('anti_labor_intensity ~ railroad_interest',
              weights=df_xs['total_labor'], data=df_xs).fit(cov_type='HC3')
wxb = smf.wls('anti_labor_intensity ~ railroad_interest + person_republican',
              weights=df_xs_pol['total_labor'], data=df_xs_pol).fit(cov_type='HC3')
wxd = smf.wls('anti_labor_intensity ~ railroad_interest + person_republican + log_county_pop',
              weights=df_xs_pop['total_labor'], data=df_xs_pop).fit(cov_type='HC3')

print(summary_col(
    [wxa, wxb, wxd],
    model_names=['(0a) Bivariate', '(0b) + Party', '(0c) + Party + Pop'],
    stars=True,
    info_dict={'N': lambda m: m.nobs, 'R²': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'person_republican', 'log_county_pop', 'Intercept']
))

Cross-sectional regressions — OLS (HC3 SEs)


                  (0a) Bivariate (0b) + Party (0c) + Party + Pop
----------------------------------------------------------------
railroad_interest 0.0895**       0.0883**     0.1124**          
                  (0.0372)       (0.0411)     (0.0490)          
person_republican                0.0676*      0.0691*           
                                 (0.0380)     (0.0396)          
log_county_pop                                -0.0497*          
                                              (0.0264)          
Intercept         0.3196***      0.2803***    0.7955***         
                  (0.0201)       (0.0326)     (0.2709)          
R-squared         0.1019         0.1481       0.2668            
R-squared Adj.    0.0855         0.1119       0.2156            
N                 57.0000        50.0000      47.0000           
R²                0.1020         0.1480       0.2670            
Standard errors in parentheses.
* p<.1, ** p

---

## 6. Coverage Amount Regression

Does railroad financial interest affect the **volume** of labor coverage?

**Outcome:** `labor_coverage_share` = labor articles / total articles per newspaper-year  
**Treatment:** `railroad_interest` (same as above)

In [13]:
# Compute total articles per newspaper-year for ISSNs in the analysis sample
conn = sqlite3.connect(DB_PATH)

analysis_issns = df['issn'].unique().tolist()
placeholders = ','.join(['?'] * len(analysis_issns))

total_articles = pd.read_sql(f"""
    SELECT issn, year, COUNT(*) as total_articles
    FROM articles
    WHERE issn IN ({placeholders})
    GROUP BY issn, year
""", conn, params=analysis_issns)

conn.close()

# Merge: labor article counts (from 'counts' df) + total articles
coverage = counts[['issn', 'year', 'total_labor']].merge(
    total_articles, on=['issn', 'year'], how='inner'
)
coverage['labor_coverage_share'] = coverage['total_labor'] / coverage['total_articles']

# Merge with treatment
df_cov = coverage.merge(paper_year_rr, on=['issn', 'year'], how='inner')
df_cov['year_str'] = df_cov['year'].astype(str)

print(f"Coverage analysis sample: {len(df_cov)} newspaper-years")
print(f"\nLabor coverage share by treatment group:")
print(df_cov.groupby('railroad_interest')['labor_coverage_share'].describe())

Coverage analysis sample: 319 newspaper-years

Labor coverage share by treatment group:
                   count      mean       std       min       25%       50%  \
railroad_interest                                                            
0                  198.0  0.001790  0.001570  0.000132  0.000644  0.001390   
1                  121.0  0.002046  0.001679  0.000120  0.000818  0.001669   

                        75%       max  
railroad_interest                      
0                  0.002420  0.009834  
1                  0.002755  0.009645  


In [14]:
# Merge person_republican and log_county_pop into coverage df
df_cov = df_cov.merge(
    df[['issn', 'year', 'person_republican', 'log_county_pop']],
    on=['issn', 'year'], how='left'
)

# Coverage regressions
c1 = smf.ols('labor_coverage_share ~ railroad_interest', data=df_cov).fit(cov_type='HC3')
c2 = smf.ols('labor_coverage_share ~ railroad_interest + C(year)', data=df_cov).fit(cov_type='HC3')

df_cov_pol = df_cov.dropna(subset=['person_republican'])
c3 = smf.ols('labor_coverage_share ~ railroad_interest + C(year) + person_republican',
             data=df_cov_pol).fit(cov_type='HC3')

df_cov_pop = df_cov.dropna(subset=['person_republican', 'log_county_pop'])
c4 = smf.ols('labor_coverage_share ~ railroad_interest + C(year) + person_republican + log_county_pop',
             data=df_cov_pop).fit(cov_type='HC3')

print(summary_col(
    [c1, c2, c3, c4],
    model_names=['(1) Bivariate', '(2) Year FE', '(3) + Party', '(4) + Party + Pop'],
    stars=True,
    info_dict={'N': lambda m: m.nobs, 'R²': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'person_republican', 'log_county_pop', 'Intercept']
))


                  (1) Bivariate (2) Year FE (3) + Party (4) + Party + Pop
-------------------------------------------------------------------------
railroad_interest 0.0003        0.0002      0.0003      -0.0001          
                  (0.0002)      (0.0002)    (0.0002)    (0.0002)         
person_republican                           0.0001      0.0001           
                                            (0.0002)    (0.0002)         
log_county_pop                                          0.0003***        
                                                        (0.0001)         
Intercept         0.0018***     0.0008***   0.0008***   -0.0025***       
                  (0.0001)      (0.0002)    (0.0003)    (0.0007)         
C(year)[T.1871]                 0.0005      0.0005      0.0004           
                                (0.0006)    (0.0007)    (0.0007)         
C(year)[T.1872]                 0.0005      0.0004      0.0004           
                                (0.00